# 07: Regulated ML Fairness, Disparate Impact (80% Rule) & SHAP Explainability

> **Lab 07 Companion Notebook**  
> **Topics**: Regulated Credit Scoring Pipeline, Deterministic Tabular Classification, Disparate Impact Ratio (EEOC 80% Rule), Demographic Parity Difference, Fairlearn MetricFrame Dashboards, SHAP Local Waterfall Attributions & Grounded Adverse Action Notices.

In [ ]:
# %% [Setup] Colab & Environment Bootstrap
import os, sys
if 'google.colab' in str(get_ipython()):
    print('Running in Google Colab. Cloning repository...')
    !git clone --depth 1 https://github.com/dip4k/senior-engineer-to-ai-engineer-roadmap.git repo
    %cd repo
    !pip install -q -r agent-forge/requirements.txt -r notebooks/requirements-notebooks.txt
    sys.path.insert(0, os.path.abspath('agent-forge'))
else:
    # Local environment setup: locate repo root and agent-forge
    repo_root = os.path.abspath('..') if os.path.exists('../agent-forge') else os.path.abspath('.')
    af_path = os.path.join(repo_root, 'agent-forge')
    if os.path.exists(af_path) and af_path not in sys.path:
        sys.path.insert(0, af_path)
    print(f'Running locally with agent-forge in path: {af_path}')

## 1. Architectural & Regulatory Foundations: Regulated Automated Decisioning

In enterprise credit underwriting, insurance risk rating, and procurement scoring, machine learning models are subject to rigorous statutory compliance standards:

1. **EU AI Act (Annex III High-Risk AI Systems)**: Mandates technical documentation, continuous bias monitoring, human oversight, and verifiable explainability for automated systems assessing creditworthiness and supplier risk.
2. **US Equal Credit Opportunity Act (ECOA) & Regulation B**: Prohibits discrimination based on protected characteristics (race, sex, age) and legally obligates creditors to provide specific, verifiable reasons (Adverse Action Notices) when denying credit. Hallucinated or generic statements ("AI determined elevated risk") constitute direct statutory violations.
3. **EEOC 80% (Four-Fifths) Rule**: The selection rate for any protected group must not fall below 80% of the selection rate for the highest group. If `Disparate Impact Ratio (DIR) < 0.80`, adverse impact is legally presumed.
4. **The Limits of 'Fairness Through Unawareness'**: Simply dropping protected attributes (`gender`, `age`) from the training feature matrix does not prevent bias because correlated proxy features (e.g., zip code, tenure, income history) readily reconstruct protected group variance.

```text
+-----------------------------------------------------------------------------------------+
|                    HYBRID REGULATED ML + GenAI DECISIONING PIPELINE                     |
+-----------------------------------------------------------------------------------------+
|                                                                                         |
|   [Applicant Data] ---> [Deterministic ML Model: Logistic / Random Forest]              |
|                                       |                                                 |
|                     [Probability & Binary Credit Decisions]                             |
|                                       |                                                 |
|            +--------------------------+--------------------------+                      |
|            |                                                     |                      |
|            v                                                     v                      |
|   [Fairness Audit Gate]                                 [SHAP Attribution Engine]       |
|   - Group Selection Rates (SR)                          - Base Value: E[f(x)]           |
|   - Disparate Impact Ratio: DIR >= 0.80                 - Local Shapley Values: phi_i   |
|   - Demographic Parity Diff: DPD <= 0.10                - Waterfall Decomposition       |
|            |                                                     |                      |
|       [Pass / Fail]                                     [Top Negative Drivers]          |
|            |                                                     |                      |
|            +--------------------------+--------------------------+                      |
|                                       |                                                 |
|                                       v                                                 |
|                 [Legally Grounded Adverse Action Synthesis]                             |
|                 - 1-to-1 Mapping to Authentic SHAP Drivers                              |
|                 - Groundedness Assertion Guardrail (Zero Hallucination)                 |
+-----------------------------------------------------------------------------------------+
```

In this notebook, we build an end-to-end pipeline: synthesizing a regulated financial dataset, training baseline classifiers, conducting a Fairlearn statistical bias audit, computing SHAP local waterfall attributions, and generating legally compliant adverse action notices.

## 2. Environment Setup & Core Imports

We import standard scientific and tabular libraries (`pandas`, `numpy`, `matplotlib`, `seaborn`, `scikit-learn`). We also import `fairlearn` and `shap` with robust graceful fallbacks to ensure compatibility across local and Google Colab environments.

In [ ]:
import os
import sys
import math
import json
from dataclasses import dataclass
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix, accuracy_score

# Optional Fairlearn import with fallback
try:
    import fairlearn
    from fairlearn.metrics import MetricFrame, selection_rate, demographic_parity_difference, demographic_parity_ratio
    HAS_FAIRLEARN = True
    print(f"Fairlearn version {fairlearn.__version__} loaded successfully.")
except ImportError:
    HAS_FAIRLEARN = False
    print("Fairlearn not installed. Using native statutory metrics implementation.")

# Optional SHAP import with fallback
try:
    import shap
    HAS_SHAP = True
    print(f"SHAP version {shap.__version__} loaded successfully.")
except ImportError:
    HAS_SHAP = False
    print("SHAP not installed. Using native linear/tree attribution engine.")

# Style configuration
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cbd5e1'
plt.rcParams['figure.autolayout'] = True

print('Scientific libraries initialized.')

## 3. Synthetic Regulated Financial Dataset Generation

We construct a synthetic dataset of 2,500 commercial loan applicants containing:

- **Financial Predictor Features (Permitted)**:
  - `annual_income`: Annual applicant revenue ($25,000 to $220,000)
  - `credit_score`: FICO credit score (350 to 850)
  - `debt_to_income_ratio`: Monthly debt obligations / Gross monthly income (0.10 to 0.65)
  - `late_payments_24m`: Count of 30+ day delinquencies in the last 24 months (0 to 6)
  - `employment_years`: Length of continuous employment or business operation (0.5 to 25.0 years)

- **Protected / Sensitive Attributes (Regulated under ECOA & EU AI Act)**:
  - `gender`: 0 = Male, 1 = Female
  - `age_group`: 0 = Under 30 (Young), 1 = 30 to 55 (Mid-career), 2 = Over 55 (Senior)
  - `minority_group`: 0 = Majority, 1 = Underrepresented Minority

- **Ground-Truth Target Variable**:
  - `loan_approved`: Binary loan approval (0 = Denied, 1 = Approved). Injected with systemic historical disparities in debt-to-income distribution to simulate real-world proxy bias.

In [ ]:
np.random.seed(42)
n_samples = 2500

# Protected Demographic Attributes
gender = np.random.binomial(n=1, p=0.48, size=n_samples)  # 0: Male, 1: Female
age_raw = np.random.normal(loc=42, scale=12, size=n_samples)
age_raw = np.clip(age_raw, 21, 75)
age_group = np.where(age_raw < 30, 0, np.where(age_raw <= 55, 1, 2))  # 0: <30, 1: 30-55, 2: >55
minority_group = np.random.binomial(n=1, p=0.32, size=n_samples)  # 0: Majority, 1: Minority

# Financial Features with Realistic Proxy Inter-correlations
# Note: Systemic wage disparities induce correlated proxies in historical tabular data
income_base = np.random.lognormal(mean=11.1, sigma=0.45, size=n_samples)
income_penalty = (gender * 0.08 + minority_group * 0.12)
annual_income = np.clip(income_base * (1.0 - income_penalty), 25000, 250000)

credit_score_base = np.random.normal(loc=680, scale=65, size=n_samples)
credit_score = np.clip(credit_score_base - (minority_group * 22) - (gender * 12), 350, 850)

dti_base = np.random.beta(a=2.5, b=4.0, size=n_samples) * 0.65 + 0.10
debt_to_income_ratio = np.clip(dti_base + (minority_group * 0.04), 0.10, 0.65)

late_payments_lambda = 0.6 + (minority_group * 0.3) + np.where(credit_score < 620, 1.2, 0.0)
late_payments_24m = np.random.poisson(lam=late_payments_lambda, size=n_samples)
late_payments_24m = np.clip(late_payments_24m, 0, 6)

employment_years = np.clip((age_raw - 20) * 0.55 + np.random.normal(0, 3, size=n_samples), 0.5, 28.0)

# Deterministic Credit Risk Scoring Function
# Log-odds of creditworthiness: high credit score & income increase odds, high DTI & late payments decrease odds
z = (
    0.012 * (credit_score - 650) +
    0.000025 * (annual_income - 65000) -
    6.5 * (debt_to_income_ratio - 0.32) -
    0.85 * late_payments_24m +
    0.06 * (employment_years - 5)
)
approval_prob = 1.0 / (1.0 + np.exp(-z))
loan_approved = (approval_prob >= 0.50).astype(int)

df_credit = pd.DataFrame({
    'applicant_id': [f'APP_{i:05d}' for i in range(1, n_samples + 1)],
    'annual_income': np.round(annual_income, 2),
    'credit_score': np.round(credit_score, 1),
    'debt_to_income_ratio': np.round(debt_to_income_ratio, 3),
    'late_payments_24m': late_payments_24m,
    'employment_years': np.round(employment_years, 1),
    'gender': gender,
    'gender_label': np.where(gender == 1, 'Female', 'Male'),
    'age_group': age_group,
    'age_label': np.where(age_group == 0, 'Under 30', np.where(age_group == 1, '30 to 55', 'Over 55')),
    'minority_group': minority_group,
    'minority_label': np.where(minority_group == 1, 'Minority', 'Majority'),
    'loan_approved': loan_approved
})

print(f"Generated {len(df_credit)} credit records.")
print(f"Overall Approval Rate: {df_credit['loan_approved'].mean() * 100:.2f}%")
df_credit.head(5)

## 4. Exploratory Data Analysis: Detecting Hidden Proxy Disparities

Before training models, compliance officers inspect the distribution of credit scores and approval rates across sensitive demographic segments. Let us examine the ground-truth historical approval rates.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

# 1. Approval Rate by Gender
sns.barplot(data=df_credit, x='gender_label', y='loan_approved', palette=['#3b82f6', '#ec4899'], ax=axes[0], capsize=0.1)
axes[0].set_title('Approval Rate by Gender', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Approval Rate', fontsize=10, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylim(0, 1.0)

# 2. Approval Rate by Age Group
sns.barplot(data=df_credit, x='age_label', y='loan_approved', palette=['#10b981', '#6366f1', '#f59e0b'], ax=axes[1], capsize=0.1)
axes[1].set_title('Approval Rate by Age Group', fontsize=12, fontweight='bold')
axes[1].set_ylabel('')
axes[1].set_xlabel('')
axes[1].set_ylim(0, 1.0)

# 3. Approval Rate by Minority Status
sns.barplot(data=df_credit, x='minority_label', y='loan_approved', palette=['#64748b', '#8b5cf6'], ax=axes[2], capsize=0.1)
axes[2].set_title('Approval Rate by Minority Status', fontsize=12, fontweight='bold')
axes[2].set_ylabel('')
axes[2].set_xlabel('')
axes[2].set_ylim(0, 1.0)

for ax in axes:
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.show()

# Tabular summary of group selection rates
print("Ground Truth Approval Rates across Demographic Segments:")
print(df_credit.groupby('gender_label')['loan_approved'].mean().round(3))
print(df_credit.groupby('age_label')['loan_approved'].mean().round(3))
print(df_credit.groupby('minority_label')['loan_approved'].mean().round(3))

## 5. Training Baseline Classifiers (Fairness Through Unawareness)

To test standard industry practices, we implement **Demographic Blindness**: we strictly omit protected attributes (`gender`, `age_group`, `minority_group`) from the training feature set `X`.

We train two baseline classifiers:
1. **Logistic Regression**: Interpretable linear risk model with calibrated log-odds coefficients.
2. **Random Forest Classifier**: Non-linear ensemble model capturing non-linear threshold boundaries.

We evaluate standard technical predictive metrics (Accuracy, ROC-AUC, Classification Report).

In [ ]:
# Prepare Permitted Feature Matrix X and Target y
feature_cols = ['annual_income', 'credit_score', 'debt_to_income_ratio', 'late_payments_24m', 'employment_years']
X = df_credit[feature_cols]
y = df_credit['loan_approved']

# Sensitive attributes preserved separately for post-prediction auditing
sensitive_df = df_credit[['gender', 'gender_label', 'age_group', 'age_label', 'minority_group', 'minority_label']]

# Train/Test Split (80% train, 20% test, stratified by approval)
X_train, X_test, y_train, y_test, sens_train, sens_test = train_test_split(
    X, y, sensitive_df, test_size=0.20, random_state=42, stratify=y
)

# Model 1: Calibrated Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

# Model 2: Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_model.fit(X_train, y_train)

# Predict on Test Set
lr_preds = lr_model.predict(X_test)
lr_probs = lr_model.predict_proba(X_test)[:, 1]

rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

print("=" * 65)
print("PREDICTIVE PERFORMANCE EVALUATION")
print("=" * 65)
print(f"Logistic Regression  - Accuracy: {accuracy_score(y_test, lr_preds) * 100:.2f}% | ROC-AUC: {roc_auc_score(y_test, lr_probs):.4f}")
print(f"Random Forest        - Accuracy: {accuracy_score(y_test, rf_preds) * 100:.2f}% | ROC-AUC: {roc_auc_score(y_test, rf_probs):.4f}")
print("=" * 65)

# Attach predictions to test dataframe for auditing
df_test_audit = X_test.copy()
df_test_audit['true_approved'] = y_test
df_test_audit['lr_pred'] = lr_preds
df_test_audit['lr_prob'] = lr_probs
df_test_audit['rf_pred'] = rf_preds
df_test_audit['rf_prob'] = rf_probs
for col in sens_test.columns:
    df_test_audit[col] = sens_test[col]

print(f"Test set prepared: {len(df_test_audit)} applicants ready for regulatory audit.")

## 6. Regulatory Fairness Auditing: Disparate Impact & The 80% Rule

High predictive accuracy does NOT guarantee regulatory compliance. We evaluate the model against statutory criteria:

### Mathematical Definitions:
1. **Group Selection Rate**: Proportion of applicants in demographic group `g` approved for credit:
   ```text
   Selection_Rate(g) = Approvals(g) / Total_Applicants(g)
   ```
2. **Disparate Impact Ratio (DIR) / Four-Fifths (80%) Rule**:
   ```text
   DIR = min(Selection_Rate(g)) / max(Selection_Rate(g))
   ```
   - **Statutory Standard**: Under the US EEOC uniform employee selection guidelines and the EU AI Act Annex III, if `DIR < 0.80`, adverse impact is legally presumed!
3. **Demographic Parity Difference (DPD)**:
   ```text
   DPD = | Selection_Rate(Group_A) - Selection_Rate(Group_B) |
   ```
   - **Standard Benchmark**: Regulators expect `DPD <= 0.10` in non-discriminatory pipelines.

Let us execute the audit across all sensitive attributes.

In [ ]:
def compute_fairness_metrics(df_sub: pd.DataFrame, group_col: str, pred_col: str) -> Dict[str, Any]:
    rates = df_sub.groupby(group_col)[pred_col].mean().to_dict()
    min_rate = min(rates.values())
    max_rate = max(rates.values())
    dir_ratio = min_rate / max_rate if max_rate > 0 else 1.0
    dpd = max_rate - min_rate
    passed_80 = (dir_ratio >= 0.80)

    return {
        'group_rates': rates,
        'min_rate': min_rate,
        'max_rate': max_rate,
        'disparate_impact_ratio': dir_ratio,
        'demographic_parity_diff': dpd,
        'passed_80_rule': passed_80
    }

# Audit Logistic Regression Predictions
audit_gender = compute_fairness_metrics(df_test_audit, 'gender_label', 'lr_pred')
audit_age = compute_fairness_metrics(df_test_audit, 'age_label', 'lr_pred')
audit_minority = compute_fairness_metrics(df_test_audit, 'minority_label', 'lr_pred')

print("=" * 75)
print("STATUTORY BIAS AUDIT SCORECARD (LOGISTIC REGRESSION BASELINE)")
print("=" * 75)
for label, audit in [('Gender', audit_gender), ('Age Group', audit_age), ('Minority Status', audit_minority)]:
    status = "PASS (Compliant)" if audit['passed_80_rule'] else "FAIL (Violation of 80% Rule)"
    print(f"{label:<16}: DIR = {audit['disparate_impact_ratio']:5.3f} | DPD = {audit['demographic_parity_diff']:5.3f} | Status: {status}")
    for grp, r in audit['group_rates'].items():
        print(f"   - {grp:<12}: Selection Rate = {r * 100:5.2f}%")
    print("-" * 75)

# Verify Fairlearn MetricFrame if available
if HAS_FAIRLEARN:
    mf_gender = MetricFrame(
        metrics={'selection_rate': selection_rate, 'accuracy': accuracy_score},
        y_true=df_test_audit['true_approved'],
        y_pred=df_test_audit['lr_pred'],
        sensitive_features=df_test_audit['gender_label']
    )
    print("\nFairlearn MetricFrame by Gender:")
    print(mf_gender.by_group)

## 7. Disparate Impact Compliance Dashboard

We plot the group selection rates alongside the **80% Statutory Threshold** line (`0.80 * max_selection_rate`). Any bar dropping below the red dashed threshold line indicates an unlawful disparate impact.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

audits = [('Gender', audit_gender, axes[0]), ('Age Group', audit_age, axes[1]), ('Minority Status', audit_minority, axes[2])]

for title, audit, ax in audits:
    groups = list(audit['group_rates'].keys())
    rates = [audit['group_rates'][g] for g in groups]
    max_r = audit['max_rate']
    thresh_80 = 0.80 * max_r

    bars = ax.bar(groups, rates, color='#3b82f6', width=0.55, edgecolor='#1e293b', linewidth=1.2)
    # Highlight failing bars in coral
    for bar, rate in zip(bars, rates):
        if rate < thresh_80:
            bar.set_color('#ef4444')

    # 80% Rule line
    ax.axhline(thresh_80, color='#dc2626', linestyle='--', linewidth=2.0, label=f'80% Threshold ({thresh_80 * 100:.1f}%)')
    ax.axhline(max_r, color='#10b981', linestyle=':', linewidth=1.5, label=f'Max Rate ({max_r * 100:.1f}%)')

    status_text = 'PASS' if audit['passed_80_rule'] else 'FAIL'
    status_color = '#10b981' if audit['passed_80_rule'] else '#ef4444'
    ax.set_title(f"{title} (DIR: {audit['disparate_impact_ratio']:.2f} - {status_text})", fontsize=12, fontweight='bold', color=status_color)
    ax.set_ylabel('Selection / Approval Rate', fontsize=10, fontweight='bold')
    ax.set_ylim(0, max(rates) * 1.35)
    ax.legend(loc='upper right', fontsize=9)
    ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 8. SHAP Local Explainability Engine: Decomposing Credit Decisions

Under the US Equal Credit Opportunity Act (ECOA) Regulation B and the EU AI Act, if an applicant is denied credit, the system **must** provide the precise financial reasons.

### Theoretical Foundation of Shapley Additive Explanations:
From cooperative game theory, the prediction `f(x)` is decomposed into additive feature contributions `phi_i`:
```text
f(x) = E[f(x)] + Σ [ phi_i ]   for i = 1 to M
```
Where:
- `E[f(x)]`: The base expected value (average approval probability across the entire applicant population).
- `phi_i`: The Shapley attribution for feature `i`. Positive `phi_i` increases approval probability; negative `phi_i` pushes the score toward denial (adverse risk factors).
- **Efficiency Axiom**: The sum of all attributions strictly equals the difference between the model output and the expected base value.

Below, we compute exact local Shapley attributions for our test applicants.

In [ ]:
# Compute Base Expected Value E[f(x)] and Feature Coefficients for Logistic Regression
base_prob = y_train.mean()
base_log_odds = np.log(base_prob / (1.0 - base_prob))

# Standardized Linear Model Shapley Vector Computation
# For logistic regression: z = intercept + sum(w_i * x_i)
mean_X = X_train.mean().values
coefs = lr_model.coef_[0]
intercept = lr_model.intercept_[0]

def compute_linear_shap(row_values: np.ndarray) -> np.ndarray:
    """
    Computes exact additive Shapley values for a linear log-odds model:
    phi_i = w_i * (x_i - E[x_i])
    """
    return coefs * (row_values - mean_X)

# If shap library is available, use shap.Explainer
if HAS_SHAP:
    explainer = shap.Explainer(lr_model, X_train)
    shap_values = explainer(X_test)
    print("SHAP Explainer computed attributions across test set.")
else:
    shap_matrix = np.array([compute_linear_shap(row) for row in X_test.values])
    print(f"Native Shapley attribution matrix computed: shape {shap_matrix.shape}")

print(f"Base Population Expected Probability E[f(x)]: {base_prob * 100:.2f}%")

## 9. Global Feature Importance: Mean Absolute SHAP Attributions

Global feature importance measures the overall influence of each feature across the entire population:
```text
Global_Importance(i) = (1 / N) * Σ | phi_i |
```
Let us visualize the global feature ranking.

In [ ]:
if HAS_SHAP:
    global_importance = np.abs(shap_values.values).mean(axis=0)
else:
    global_importance = np.abs(shap_matrix).mean(axis=0)

df_importance = pd.DataFrame({
    'feature': feature_cols,
    'mean_abs_shap': global_importance
}).sort_values('mean_abs_shap', ascending=True)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.barh(df_importance['feature'], df_importance['mean_abs_shap'], color='#6366f1', edgecolor='#1e293b', height=0.55)
ax.set_title('Global Feature Importance (Mean Absolute SHAP Value)', fontsize=12, fontweight='bold', pad=12)
ax.set_xlabel('Mean |SHAP Value| (Impact on Model Log-Odds)', fontsize=10, fontweight='bold')
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.show()

print("Global Risk Driver Ranking:")
for _, r in df_importance.sort_values('mean_abs_shap', ascending=False).iterrows():
    print(f"  - {r['feature']:<25}: {r['mean_abs_shap']:.4f}")

## 10. Local Decision Decomposition: Approved vs Rejected Applicants

Under ECOA Regulation B, we must isolate **individual adverse drivers** for any applicant whose loan was denied.

We compare two contrasting test applicants:
1. **Applicant A (Approved)**: High credit score and healthy liquidity push the probability above the threshold.
2. **Applicant B (Denied)**: Elevated debt-to-income and late payments drive the score into rejection.

Below, we build a custom **Matplotlib SHAP Waterfall Plot** showing the step-by-step contribution of each feature moving from the base value `E[f(x)]` to the final prediction `f(x)`.

In [ ]:
# Identify an Approved and a Denied applicant from the test set
denied_idx = df_test_audit[df_test_audit['lr_pred'] == 0].index[0]
approved_idx = df_test_audit[df_test_audit['lr_pred'] == 1].index[0]

def plot_shap_waterfall(applicant_row: pd.Series, applicant_id: str, title: str):
    features = feature_cols
    x_vals = applicant_row[features].values.astype(float)
    phis = compute_linear_shap(x_vals)

    # Sort features by absolute contribution
    sort_indices = np.argsort(np.abs(phis))
    sorted_feats = [features[i] for i in sort_indices]
    sorted_phis = phis[sort_indices]
    sorted_x = x_vals[sort_indices]

    # Starting base log-odds
    curr = base_log_odds
    starts = []
    for p in sorted_phis:
        starts.append(curr)
        curr += p
    final_log_odds = curr
    final_prob = 1.0 / (1.0 + np.exp(-final_log_odds))

    fig, ax = plt.subplots(figsize=(11, 5.5))
    y_pos = np.arange(len(sorted_feats))

    for i, (feat, phi, start, val) in enumerate(zip(sorted_feats, sorted_phis, starts, sorted_x)):
        color = '#10b981' if phi >= 0 else '#ef4444'
        bar_left = min(start, start + phi)
        bar_width = abs(phi)
        ax.barh(y=i, width=bar_width, left=bar_left, color=color, edgecolor='#1e293b', height=0.55)

        # Value label
        sign = '+' if phi >= 0 else ''
        ax.text(
            bar_left + bar_width + 0.05,
            i,
            f"{sign}{phi:.2f} (val: {val:,.2f})",
            va='center',
            ha='left',
            fontsize=9,
            fontweight='bold'
        )

    # Reference lines
    ax.axvline(base_log_odds, color='#64748b', linestyle=':', linewidth=1.5, label=f'Base Value E[f(x)] = {base_prob * 100:.1f}%')
    ax.axvline(0.0, color='#1e293b', linestyle='-', linewidth=1.0, alpha=0.5, label='Decision Threshold (prob = 50%)')

    ax.set_yticks(y_pos)
    ax.set_yticklabels(sorted_feats, fontsize=10, fontweight='bold')
    decision_label = 'APPROVED' if final_prob >= 0.50 else 'DENIED'
    title_color = '#10b981' if final_prob >= 0.50 else '#ef4444'
    ax.set_title(f"{title} ({applicant_id}) - {decision_label} (Prob: {final_prob * 100:.1f}%)", fontsize=12, fontweight='bold', color=title_color)
    ax.set_xlabel('Log-Odds Contribution (phi_i)', fontsize=10, fontweight='bold')
    ax.legend(loc='lower right', framealpha=0.9)
    ax.grid(axis='x', linestyle='--', alpha=0.5)
    plt.show()

    # Return top negative risk drivers
    adverse_drivers = [
        (features[i], phis[i], x_vals[i])
        for i in np.argsort(phis) if phis[i] < 0
    ]
    return adverse_drivers

print("Plotting SHAP waterfall for Denied Applicant:")
denied_row = df_test_audit.loc[denied_idx]
adverse_drivers = plot_shap_waterfall(denied_row, denied_row['applicant_id'], 'SHAP Adverse Action Decomposition')

print("Top Adverse Drivers (Reasons for Rejection):")
for feat, impact, val in adverse_drivers:
    print(f"  - {feat:<25}: Impact = {impact:6.2f} (Applicant Value: {val:,.2f})")

## 11. Legally Compliant Adverse Action Notice Generation with Guardrails

Under the US Equal Credit Opportunity Act (ECOA) Regulation B and the EU AI Act, creditors must issue an Adverse Action Notice stating the **principal reasons** for denial. These reasons must map **strictly** to the authentic top negative SHAP features.

### Groundedness Assertion Guardrail:
When an LLM synthesizes the formal letter, our automated guardrail verifies that:
1. Every cited factor is present in the authentic top negative SHAP feature set.
2. The notice does NOT invent hallucinated, unverified reasons (e.g. "insufficient collateral" or "short business tenure").
3. Any draft failing the groundedness gate is quarantined immediately and replaced with a deterministic fallback template.

In [ ]:
@dataclass
class AdverseActionVerification:
    is_grounded: bool
    authorized_factors: List[str]
    cited_factors: List[str]
    hallucinated_factors: List[str]
    letter_text: str

class AdverseActionGuardrail:
    """
    Zero-Trust Regulatory Guardrail asserting that generated adverse action letters
    contain only authentic SHAP-derived financial drivers.
    """
    AUTHORIZED_FEATURE_DICTIONARY = {
        'debt_to_income_ratio': ['debt-to-income', 'debt to income', 'high debt ratio', 'monthly debt obligations'],
        'late_payments_24m': ['delinquencies', 'late payments', 'past due accounts', 'late payment history'],
        'credit_score': ['credit score', 'fico score', 'bureau score', 'low credit rating'],
        'annual_income': ['income level', 'annual income', 'insufficient revenue', 'low earnings'],
        'employment_years': ['employment history', 'job tenure', 'short employment', 'time in role']
    }

    UNAUTHORIZED_HALLUCINATIONS = [
        'collateral', 'business age', 'guarantor', 'bankruptcy', 'foreclosure', 'inquiry volume', 'criminal record'
    ]

    def verify(self, letter_text: str, top_shap_negative_features: List[str]) -> AdverseActionVerification:
        text_lower = letter_text.lower()
        cited = []
        hallucinated = []

        # Check authorized factors
        for feat, synonyms in self.AUTHORIZED_FEATURE_DICTIONARY.items():
            if any(syn in text_lower for syn in synonyms):
                cited.append(feat)
                if feat not in top_shap_negative_features:
                    hallucinated.append(f"Unjustified factor cited: '{feat}' is not among top negative drivers.")

        # Check unauthorized hallucinated terms
        for term in self.UNAUTHORIZED_HALLUCINATIONS:
            if term in text_lower:
                hallucinated.append(f"Unauthorized / Hallucinated credit denial reason: '{term}'")

        is_grounded = (len(hallucinated) == 0 and len(cited) > 0)
        return AdverseActionVerification(
            is_grounded=is_grounded,
            authorized_factors=top_shap_negative_features,
            cited_factors=cited,
            hallucinated_factors=hallucinated,
            letter_text=letter_text
        )

# Top negative drivers from our SHAP analysis
top_adverse_keys = [feat for feat, impact, val in adverse_drivers[:2]]
print(f"Permitted regulatory denial factors for applicant: {top_adverse_keys}")

# Case 1: Valid Grounded Adverse Action Notice
valid_notice = (
    f"Dear Applicant {denied_row['applicant_id']},\n\n"
    "Thank you for your recent commercial credit application. Under the Equal Credit Opportunity Act (ECOA),\n"
    "we are providing the primary verifiable financial factors that influenced our credit decision:\n\n"
    f"1. Elevated Debt-to-Income Ratio (Applicant reported: {denied_row['debt_to_income_ratio'] * 100:.1f}%).\n"
    f"2. Delinquencies in the Last 24 Months (Applicant recorded: {int(denied_row['late_payments_24m'])} late payments).\n\n"
    "You have the right to request a disclosure of your credit file under federal law.\n"
    "Sincerely,\nRegulated Credit Decisioning Office"
)

# Case 2: Hallucinated Adverse Action Notice (Violates Regulation B)
hallucinated_notice = (
    f"Dear Applicant {denied_row['applicant_id']},\n\n"
    "Your credit application was declined due to insufficient collateral and high volume of recent credit inquiries.\n"
    "Sincerely,\nAutomated AI System"
)

guardrail = AdverseActionGuardrail()
res_valid = guardrail.verify(valid_notice, top_adverse_keys)
res_invalid = guardrail.verify(hallucinated_notice, top_adverse_keys)

print("=" * 75)
print("ADVERSE ACTION NOTICE REGULATORY GUARDRAIL VERIFICATION")
print("=" * 75)
print(f"Case 1 (Grounded Notice)     -> Status: {'APPROVED' if res_valid.is_grounded else 'REJECTED'}")
print(f"   Factors Cited: {res_valid.cited_factors}")
print(f"Case 2 (Hallucinated Notice)  -> Status: {'APPROVED' if res_invalid.is_grounded else 'REJECTED (QUARANTINED)'}")
print(f"   Violations: {res_invalid.hallucinated_factors}")
print("=" * 75)

## 12. Fairness Mitigation Strategies & Hands-On Engineering Katas

### Enterprise Mitigation Strategies for Disparate Impact:
1. **Pre-Processing (Fair Data Engineering)**: Re-weighing sample weights to balance representation before model training.
2. **In-Processing (Constrained Optimization)**: Using Fairlearn's `ExponentiatedGradient` with a Demographic Parity or Equalized Odds constraint.
3. **Post-Processing (Calibrated Threshold Optimization)**: Selecting demographic-specific decision thresholds `tau_g` such that selection rates equalize across groups while minimizing accuracy loss.

---

### 🥋 Hands-On Katas for the Senior Engineer:
1. **Task 1: Equalized Odds Audit (True Positive Rate Parity)**
   - Calculate the True Positive Rate (Sensitivity) across gender and age groups: `TPR = TP / (TP + FN)`. Does the model approve qualified women at the same rate as qualified men?
2. **Task 2: Implement Post-Processing Threshold Optimization**
   - Calculate distinct decision thresholds for Majority and Minority applicants to bring the Disparate Impact Ratio above 0.85.
3. **Task 3: Counterfactual Fairness Testing**
   - Write a unit test that takes an applicant record, toggles the protected attribute `gender` from 0 to 1, and asserts that the predicted probability variance `|p_male - p_female|` remains below 0.02.